# Test estilométricos clásicos

Este notebook se centra en la aplicación de técnicas de estilometría clásicas para analizar los textos de los autores seleccionados. Estas técnicas incluyen:

1. El **test de las curvas características de composición de Mendenhall** (_Mendenhall’s Characteristic Curves of Composition_)
2. El **método del chi-cuadrado de Kilgariff** (_Kilgariff’s Chi-Squared Method_)
3. El **método de la distancia de Burrows** (_Burrows’s Delta Method_)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from whodunit_stylometry.constants import AUTHORS_ABREV_MAP, AUTHORS_NORM_MAP
from whodunit_stylometry.utils.data_utils import load_corpus_by_dataframe
from whodunit_stylometry.utils.nlp_utils import (
    fix_gutenberg_linebreaks,
    normalize_text_for_tokenization,
    CustomTokenizer,
    is_alpha_tokens,
)
from whodunit_stylometry.utils.stats_utils import (
    word_length_distributions_random_blocks,
    align_distributions,
    distance_matrix,
    compute_average_curve,
    classify_test_works_by_average_curve,
)
from whodunit_stylometry.utils.plot_utils import plot_word_length_distributions

## Carga del corpus

In [ ]:
# Configuración
SEED = 1
CORPUS_PATH = Path("../corpus")

In [ ]:
# Cargamos los metadatos del corpus y añadimos la
# ruta a cada archivo de texto. Esto nos permitirá
# más adelante leer los textos más fácilmente.
df_meta = pd.read_csv(CORPUS_PATH / "corpus_metadata.csv", sep=";")
df_meta["file_path"] = df_meta.apply(
    lambda x: CORPUS_PATH / "hand_cleaned" / AUTHORS_NORM_MAP[x["author"]] / x["file_name"], axis=1
)
print(df_meta.shape)
df_meta.head()

In [ ]:
# Nos quedamos solo con los textos de las novelas de misterio
# ya que el género influye mucho en algunos de los tests
# estilométricos clásicos que vamos a realizar a continuación.
df_meta_f = df_meta[df_meta["genre"] == "mystery"].copy()
df_meta_f.shape

In [ ]:
# Nos quedamos con una muestra de 1 texto por
# autor para probar la efectividad de los tests
# para identificar al autor de un texto desconocido.
df_test = df_meta_f.groupby("author", group_keys=False).sample(n=1, random_state=SEED)

# Original sin las obras de prueba seleccionadas
df_meta_f = df_meta_f.drop(index=df_test.index)

df_test = df_test.reset_index(drop=True)
df_meta_f = df_meta_f.reset_index(drop=True)

In [ ]:
# Cargamos el corpus de los autores en un diccionario
corpus_by_author = load_corpus_by_dataframe(df_meta_f)
corpus_by_author.keys()

In [ ]:
# Contamos el número de textos por autor para
# ver si el dataset está balanceado o no
df_meta["author"].value_counts()

In [ ]:
# Calculamos la longitud de cada corpus por autor en caracteres
lengths = [len(corpus_by_author[author]) for author in AUTHORS_ABREV_MAP.keys()]

corpus_lengths_df = pd.DataFrame({"author": AUTHORS_ABREV_MAP.values(), "length_in_chars": lengths}).sort_values(
    "length_in_chars", ascending=False
)
corpus_lengths_df

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=corpus_lengths_df, x="author", y="length_in_chars")
plt.xlabel("Corpus")
plt.ylabel("Longitud (nº de caracteres)")
plt.title("Longitud de cada corpus en caracteres")
plt.show()

Como vemos, el dataset está bastante desbalanceado en cuanto a número de caracteres/obras por autor, lo que puede afectar a la efectividad de los tests estilométricos.

In [ ]:
# Mostramos los primeros 200 caracteres de
# cada corpus para verificar su contenido
for author in AUTHORS_ABREV_MAP.keys():
    print(corpus_by_author[author][:200])
    print("=" * 40)

## Limpieza y normalización del texto

Para más información sobre la limpieza y normalización del texto, consultar el apéndice correspondiente en la memoria (`Apéndice II - Normalización de los textos`).

In [ ]:
# Normalizamos el texto de cada corpus
corpus_by_author_norm = {}

for author in AUTHORS_ABREV_MAP.keys():
    normalized_text = normalize_text_for_tokenization(fix_gutenberg_linebreaks(corpus_by_author[author]))
    corpus_by_author_norm[author] = normalized_text
    unique_chars = set(normalized_text)
    print(f"{author}: {unique_chars} ({len(unique_chars)})")

In [ ]:
for author in AUTHORS_ABREV_MAP.keys():
    print(corpus_by_author_norm[author][:200])
    print("=" * 40)

Parece que la limpieza y normalización del texto se ha realizado correctamente, no vemos caracteres extraños ni problemas evidentes en el texto.

## Tokenización de los textos

Para más información sobre la tokenización de los textos, consultar el apéndice correspondiente en la memoria (`Apéndice II - Normalización de los textos`).

In [ ]:
# Tokenizamos cada corpus por autor y lo
# guardamos en un nuevo diccionario
tokenizer = CustomTokenizer()

corpus_by_author_tokens = {}
for author, norm_text in corpus_by_author_norm.items():
    toks = []
    tokens_all = tokenizer.tokenize(norm_text)
    tokens_alpha = is_alpha_tokens(tokens_all, keep_alpha=True)
    toks.extend(tokens_alpha)
    corpus_by_author_tokens[author] = toks
    print(f"{author}: {toks[:20]} ... ({len(toks)} tokens)")

## 1. Curvas características de composición de Mendenhall

Con el fin de estudiar la estabilidad interna del estilo de cada autor, en este apartado se emplean las **curvas características de composición de Mendenhall**, construidas a partir de la distribución de longitudes de palabra. La idea de base es que cada texto puede caracterizarse, al menos parcialmente, por la proporción de palabras de longitud 1, 2, 3, etc., y que este patrón puede actuar como una huella estilística.

Para cada autor, el procedimiento parte de su corpus ya tokenizado y extrae varios bloques contiguos aleatorios de igual tamaño. En este caso, se toman 50 bloques de 100.000 tokens cada uno. Sobre cada bloque se calcula la distribución de longitudes de palabra, es decir, se cuenta cuántas palabras tienen longitud 1, cuántas longitud 2, y así sucesivamente. Posteriormente, esos conteos se normalizan, de modo que cada bloque queda representado por una distribución de frecuencias relativas.

Cada una de estas distribuciones constituye una curva de Mendenhall para un bloque concreto del corpus. Como no todos los bloques contienen necesariamente palabras de todas las longitudes posibles, las distribuciones se alinean sobre un mismo rango fijo de longitudes. Así, cada bloque queda representado por un vector comparable con los demás, donde cada posición corresponde a una longitud de palabra determinada y las ausencias se rellenan con valor cero.

Una vez obtenidas las curvas alineadas, se calcula la **distancia de Jensen-Shannon** entre todos los pares de bloques del mismo autor. Esta medida cuantifica el grado de diferencia entre dos distribuciones de probabilidad: valores próximos a cero indican curvas muy parecidas, mientras que valores más altos reflejan una mayor divergencia. A partir de la matriz de distancias resultante, se resume la variabilidad interna del autor mediante dos estadísticas: la **media** y la **desviación típica** de las distancias.

La interpretación de estas medidas es directa. Una **distancia media baja** sugiere que los distintos fragmentos del corpus del autor presentan distribuciones de longitudes de palabra muy similares entre sí, lo que apunta a una mayor homogeneidad estilística desde este punto de vista. Por el contrario, una **distancia media alta** indica una mayor variación interna. La desviación típica, por su parte, informa sobre la regularidad de esa variación: valores bajos señalan un comportamiento más uniforme entre bloques, mientras que valores altos revelan que algunas partes del corpus se parecen mucho entre sí y otras difieren de manera más acusada.

Este análisis nos va permitir evaluar hasta qué punto la curva de Mendenhall de un autor se mantiene estable a lo largo de diferentes fragmentos de su obra. Así, no solo permite describir cómo se distribuyen las longitudes de palabra, sino también medir numéricamente hasta qué punto ese patrón se mantiene estable dentro del texto.

In [ ]:
# Primero, y por curiosidad, vamos a calcular la
# longitud de la palabra más larga por autor
longest_word_by_author = {
    author: (max(tokens, key=len), len(max(tokens, key=len))) for author, tokens in corpus_by_author_tokens.items()
}
longest_word_by_author

Nos podría llamar la atención la palabra más larga de A. K. Green, pero comprobamos que no se trata de un error de OCR o formato, sino que realmente forma parte de una de sus obras (*The Golden Slipper and Other Problems for Violet Strange*). El resto son palabras recogidas en el diccionario de la lengua inglesa.

In [ ]:
# Diccionario para guardar las distribuciones de
# longitudes de palabra de los bloques por autor
author_distributions = {}
# Diccionario para guardar las estadísticas (media
# y desviación) de las distribuciones de cada autor
author_stats = {}

BLOCK_SIZE = 100_000
N_BLOCKS = 50

for author in AUTHORS_ABREV_MAP.keys():
    # Calculamos las distribuciones de longitud de palabra
    # de los bloques aleatorios del corpus del autor
    distributions = word_length_distributions_random_blocks(
        corpus_by_author_tokens[author],
        block_size=BLOCK_SIZE,
        n_blocks=N_BLOCKS,
        seed=SEED,
    )

    author_distributions[author] = distributions

    # Dibujamos las distribuciones de longitud de palabra
    # de los bloques del autor
    plot_word_length_distributions(AUTHORS_ABREV_MAP[author], distributions, block_size=BLOCK_SIZE)

    # Alineamos las distribuciones para que tengan
    # la misma longitud en todos los bloques del autor
    aligned, lengths = align_distributions(distributions)
    # Calculamos la matriz de distancias entre los bloques
    D = distance_matrix(aligned)
    # Nos quedamos solo con el triángulo superior (sin diagonal)
    distances = D[np.triu_indices_from(D, k=1)]

    # Calculamos la media y desviación de las distancias
    # entre las distribuciones de los bloques del autor
    mean_distance = np.mean(distances)
    std_distance = np.std(distances)

    # Guardamos resultados
    author_stats[author] = {
        "mean": mean_distance,
        "std": std_distance,
    }

    print(f"{author}")
    print(f"  Mean: {mean_distance:.5f}")
    print(f"  Std:  {std_distance:.5f}")

In [ ]:
# Guardamos las estadísticas en un DataFrame
# y mostramos el resultado para comparar entre autores
author_stats_df = pd.DataFrame.from_dict(author_stats, orient="index").reset_index().rename(columns={"index": "author"})

author_stats_df

Los resultados muestran diferencias claras en la estabilidad interna de las curvas de Mendenhall entre autores. Arthur Morrison y Arthur Conan Doyle presentan la menor distancia media y la menor desviación típica, lo que indica que la distribución de longitudes de palabra de sus distintos bloques son las más homogéneas y regulares del conjunto.

En el extremo opuesto, Anna Katharine Green y Richard Austin Freeman registran las mayores distancias medias y desviaciones típicas, lo que sugiere una mayor variabilidad interna en este rasgo estilístico. Arthur Morrison, Gilbert Keith Chesterton y Wilkie Collins ocupan posiciones intermedias, con curvas relativamente estables, aunque menos uniformes que las de Morrison y Doyle.

Los resultados permiten ordenar a los autores, de mayor a menor homogeneidad interna, aproximadamente así:

**Arthur Morrison > Arthur Conan Doyle > Gilbert Keith Chesterton > Wilkie Collins > Richard Austin Freeman > Anna Katharine Green**

Esto no significa que unos autores “escriban mejor” que otros, ni tampoco que sus estilos sean más simples o más complejos. Lo que indica, de forma más precisa, es que la distribución de longitudes de palabra se mantiene con distinta estabilidad a lo largo de sus corpus. En unos casos, esa distribución funciona como una huella muy consistente y, en otros, presenta más fluctuaciones entre fragmentos.

### Curva característica media por autor

Una vez obtenidas las curvas de Mendenhall para los distintos bloques aleatorios de cada autor, el siguiente paso consiste en construir una curva característica media que resuma su comportamiento global. El objetivo ahora es disponer de una representación única por autor, calculada a partir de la información contenida en todos los bloques previamente muestreados.

La función `compute_average_curve()` recibe como entrada una lista de distribuciones de longitudes de palabra, donde cada distribución corresponde a uno de los bloques aleatorios del corpus del autor. Cada una de estas distribuciones está representada como un diccionario en el que las claves son las longitudes de palabra y los valores sus frecuencias relativas dentro del bloque.

La función recorre todas las distribuciones y acumula, para cada longitud de palabra, la suma de sus frecuencias relativas en los distintos bloques. Una vez agregadas todas las frecuencias, la suma acumulada para cada longitud se divide entre el número total de bloques. De este modo, se obtiene una frecuencia relativa media para cada longitud de palabra, es decir, una estimación del comportamiento promedio de la curva de Mendenhall del autor a lo largo de todo su corpus.

Esta curva media puede entenderse como una versión suavizada y representativa de las curvas individuales por bloque. Mientras que las curvas obtenidas antes permitían estudiar la variación interna de cada autor, la curva media resume su comportamiento general y atenúa el efecto de pequeñas variaciones o irregularidades presentes en fragmentos concretos.

Una vez calculada la curva media de cada autor, estas se almacenan en el diccionario `average_curves`. A continuación, se recopilan todas las longitudes de palabra observadas en el conjunto de autores para construir un eje horizontal común. Este paso garantiza que todas las curvas se representen sobre la misma escala y que puedan compararse visualmente de forma directa, incluso si algunos autores no presentan determinadas longitudes en su curva media.

Finalmente, mostramos una representación gráfica en la que:

- el eje `x` corresponde a la longitud de palabra
- el eje `y` representa la frecuencia relativa media
- cada línea corresponde a la curva característica media de un autor

La visualización resultante permite comparar los perfiles medios de composición de los distintos autores y detectar posibles semejanzas o diferencias en la distribución de longitudes de palabra. En este sentido, mientras que el análisis anterior se centraba en la estabilidad interna de cada autor, esta representación permite una comparación entre autores más directa, al condensar cada corpus en una única curva media.

In [ ]:
average_curves = {}

# Calculamos la curva media por autor
for author in AUTHORS_ABREV_MAP.keys():
    distributions = author_distributions[author]
    average_curve = compute_average_curve(distributions)
    average_curves[author] = average_curve

# Definimos un eje x común (todas las longitudes
# que aparezcan en cualquier autor)
all_lengths = sorted({L for curve in average_curves.values() for L in curve.keys()})

# Visualizamos todas las curvas en la misma figura
plt.figure(figsize=(10, 6))

for author, curve in average_curves.items():
    x = all_lengths
    y = [curve.get(L, 0.0) for L in x]
    label = AUTHORS_ABREV_MAP.get(author, author)
    plt.plot(x, y, label=label, marker="o", alpha=0.9)

plt.xlabel("Longitud de palabra")
plt.ylabel("Frecuencia relativa media")
plt.title("Curvas características (media por autor)")
plt.grid(True)
plt.legend(title="Autor", loc="upper right")
plt.tight_layout()
plt.show()

Por último, comparamos cuantitativamente entre sí las curvas medias de Mendenhall de todos los autores. Para ello, primero alineamos las distribuciones para que todas queden representadas sobre el mismo rango de longitudes de palabra. Después, calculamos la distancia de Jensen-Shannon entre cada par de curvas medias, obteniendo una matriz de distancias. Finalmente, transformamos esa matriz en un DataFrame etiquetado con los nombres de los autores donde los valores más bajos indican autores con perfiles medios más parecidos, mientras que los más altos señalan una mayor diferencia en la composición de la longitud de palabra entre los autores.

In [ ]:
# Alineamos las curvas medias para que tengan la misma longitud
aligned, lengths = align_distributions(average_curves.values())

# Calculamos la matriz de distancias entre las curvas medias
D = distance_matrix(aligned)

# Transformamos la matriz de distancias en un DataFrame
# etiquetado con los nombres de los autores
pd.DataFrame(D, columns=AUTHORS_ABREV_MAP.values(), index=AUTHORS_ABREV_MAP.values())

La comparación entre las curvas medias de Mendenhall muestra que algunos autores presentan patrones más parecidos entre sí que otros. Los valores más bajos de la matriz de distancias se observan entre Arthur Morrison y G. K. Chesterton, así como entre Arthur Morrison y Arthur Conan Doyle, lo que indica que estos autores presentan distribuciones medias de longitud de palabra relativamente similares. Anna Katharine Green también se sitúa cerca de este grupo, con distancias moderadas respecto a Doyle y Morrison.

Por el contrario, Richard Austin Freeman y Wilkie Collins forman un par relativamente próximo entre sí, pero más alejado del resto de autores. En particular, Freeman registra algunas de las distancias más altas de la matriz, especialmente con Doyle, lo que sugiere un patrón estilístico bastante diferente según esta métrica.

Estos resultados indican que las curvas características de composición de Mendenhall permiten identificar afinidades parciales entre autores y detectar ciertos patrones de agrupamiento, aunque no separaciones absolutas. Así, parece que la distribución media de longitudes de palabra se revela como un rasgo útil para comparar perfiles estilísticos globales dentro del corpus, pero con limitaciones en su capacidad para discriminar de forma tajante entre autores.

### Test de atribución basado en las curvas de Mendenhall

Aún teniendo en cuenta lo limitado que es el análisis de las curvas características de composición de Mendenhall para discriminar entre autores, a continuación proponemos un test de atribución basado en esta métrica. El objetivo es evaluar si la distancia de Jensen-Shannon entre la curva media de un autor y la curva media de un texto desconocido puede servir como criterio para atribuir dicho texto a uno de los autores del corpus.

Para ello, vamos a utilizar los textos de cada autor que habíamos apartado y no habían sido incluidos en el conjunto de bloques utilizados para calcular las curvas medias. Estos textos, que forman el conjunto de prueba, se van a procesar de la misma manera que los bloques anteriores para obtener su distribución de longitudes de palabra.

A continuación, se calculará la distancia de Jensen-Shannon entre la curva media de cada autor y la curva del texto desconocido y, el autor cuya curva media presente la distancia más baja con respecto al texto desconocido, se considerará el candidato más probable para la atribución.

Por último, se evaluará la precisión de este test de atribución comprobando si el autor asignado coincide con el autor real del texto desconocido.

In [ ]:
# Cargamos los textos del conjunto de prueba
corpus_by_author_test = load_corpus_by_dataframe(df_test)
corpus_by_author_test.keys()

In [ ]:
# Normalizamos y tokenizamos los textos del conjunto de prueba
# igual que hicimos con los textos del corpus de cada autor
author_test_tokens = {}

for author, text in corpus_by_author_test.items():
    toks = []
    tokens_all = tokenizer.tokenize(normalize_text_for_tokenization(fix_gutenberg_linebreaks(text)))
    tokens_alpha = is_alpha_tokens(tokens_all, keep_alpha=True)
    toks.extend(tokens_alpha)
    author_test_tokens[author] = toks
    print(f"{author}: {toks[:20]} ... ({len(toks)} tokens)")

In [ ]:
# Asignamos las tokens de los textos de prueba al
# DataFrame para poder trabajar con ellos
df_test["tokens"] = df_test["author"].map(author_test_tokens)
df_test["tokens"]

In [ ]:
# Clasificamos los textos de prueba por autor utilizando
# la curva media de cada autor y la distancia mínima
# entre distribuciones de longitud de palabra como
# criterio de clasificación
results_df = classify_test_works_by_average_curve(
    df_test=df_test,
    average_curves=average_curves,
    token_col="tokens",
    title_col="title",
    true_author_col="author",
    block_size=5000,
    n_blocks=20,
    max_len=20,
    seed=SEED,
)

results_df

In [ ]:
results_df["correct"] = results_df["true_author"] == results_df["pred_author"]
results_df[["title", "true_author", "pred_author", "min_distance", "correct"]]

In [ ]:
accuracy = results_df["correct"].mean()
print(f"Accuracy: {accuracy:.2%}")

Como vemos, para estos corpus el test de atribución basado en las curvas de Mendenhall funciona sorprendentemente bien, con una precisión del 100% en la clasificación de los textos de prueba. Esto sugiere que, a pesar de las limitaciones conocidas de esta técnica, la distribución de longitudes de palabra puede actuar como una huella estilística suficientemente distintiva para discriminar entre los autores seleccionados en este caso concreto.

Esto no implica que esta técnica sea infalible o que funcione igual de bien en otros contextos. A pesar de que aquí ha demostrado ser un método útil para la atribución de autoría hay que tomar estos resultados con cautela, ya que podrían no generalizarse a otros conjuntos de autores o textos.

## 2. Método del chi-cuadrado de Kilgariff